# Sample scenario — learning notebook

**Milestone 1: understand a scenario before executing it.**

We will use this notebook to learn one concept at a time. This first lesson contains explanations, a proposed scenario input contract, and one runnable Python exercise that prints a sample request. App observations and test results have not been collected. Model calls and app control come in later lessons.

The approved workflow is **Executor → Analyzer → Documenter → Reporter**, with conditional human-in-the-loop pauses. The scenario requirements below are for review; the rest of the server is not being implemented here.

Source of truth: [server requirements](../docs/requirements.md), especially Sections 5 and 6.

## This notebook

Part **01 — Scenario input and agent workflow** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** No app/session activity.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.


## 1. Start with the human's intent

> In Movies, search for Interstellar, open the 2014 movie, and verify that its details show the title “Interstellar” and release year 2014.

This is our agreed first concrete scenario, using the existing Movies client. Other applications can supply their own scenarios and profiles; the testing engine remains domain-independent. The dictionary in this lesson is specific to this example, rather than the final generic server schema.

The sentence contains two different things:

- **Requested tasks:** search for `Interstellar`, identify the 2014 movie, and open its details. The UI route for doing this is determined later from app context and fresh observations.
- **Assertions:** the details screen shows the title `Interstellar` and release year `2014`.

The human has identified the app and target movie. The server still needs a configured execution profile before controlling the app.

### First exercise: make the input visible

Run the next cell using the server's Python kernel. It uses only Python's standard library.

- `scenario_text` holds the agreed English instruction from Section 1.
- `scenario_input` is a **hand-constructed learning example** of the information needed for this scenario.
- The requested app, business-level tasks, movie identity, and expected title/year come from the instruction. `requested_actions` describes those tasks; it does not contain executable taps, text-entry commands, or selectors.
- `application_profile_ref` stays unresolved until we define the server's run profile. An app name alone does not identify its installed build or device session.

The exercise proposes exact title comparison and records that choice explicitly. It does not claim that the app has displayed the expected title or year.

In the eventual system, the human supplies English and selects an app profile. The system extracts and validates the relevant information; the user does not fill out this dictionary manually. We are constructing it once to make the interpretation visible before adding an LLM.

In [1]:
from pprint import pprint

scenario_text = (
    "In Movies, search for Interstellar, open the 2014 movie, "
    "and verify that its details show the title Interstellar "
    "and release year 2014."
)

scenario_input = {
    "instruction": scenario_text,
    "application_name": "Movies",
    "application_profile_ref": None,
    "requested_actions": [
        "search_for_movie",
        "open_matching_movie_details",
    ],
    "search_query": "Interstellar",
    "target_item_identity": {"title": "Interstellar", "release_year": 2014},
    "expected_title": "Interstellar",
    "expected_release_year": 2014,
    "title_matching_rule": "exact",  # Proposed comparison rule for this exercise.
}

pprint(scenario_input, sort_dicts=False)


{'instruction': 'In Movies, search for Interstellar, open the 2014 movie, and '
                'verify that its details show the title Interstellar and '
                'release year 2014.',
 'application_name': 'Movies',
 'application_profile_ref': None,
 'requested_actions': ['search_for_movie', 'open_matching_movie_details'],
 'search_query': 'Interstellar',
 'target_item_identity': {'title': 'Interstellar', 'release_year': 2014},
 'expected_title': 'Interstellar',
 'expected_release_year': 2014,
 'title_matching_rule': 'exact'}


### Read the output, then pause

The search query, movie identity, and expected title/year come from the agreed scenario. `application_profile_ref` remains `None`, meaning **the technical run profile is unresolved**. Naming Movies does not configure its bundle ID, Simulator, driver endpoint, or backend environment.

**Requested tasks and UI actions are different.** “Search for Interstellar” is a task extracted from the English request. Whether to tap a Search tab, open a search icon, or use a search field already on screen depends on app knowledge and the current UI. The earlier assumed `open_search` entry has been removed. The executor will choose actual tool actions after observing the app.

| Information | What it identifies | Available at this stage? |
| --- | --- | --- |
| `target_item_identity` | The business entity: Interstellar, released in 2014 | Yes, from the human request. |
| `application_profile_ref` | The app/environment configuration used for execution | Still unresolved in this lesson. |
| UI locator, including a semantics/accessibility identifier when available | A control such as the search field or a particular result | Not present in this sample input. Ground actual controls during execution. |

The executor will combine the selected profile, relevant approved app documentation or procedures, and **fresh UI hierarchy/screenshots from tools** to decide its next action. Known locator hints may be supplied in app knowledge, but must be checked against the current app. It does not need every screen or identifier in advance.

For Flutter, a developer-assigned semantics `identifier` appears in the native accessibility hierarchy; on iOS it becomes an accessibility identifier. See the [Flutter identifier documentation](https://api.flutter.dev/flutter/semantics/SemanticsProperties/identifier.html). Appium's [XCUITest locator strategies](https://appium.github.io/appium-xcuitest-driver/latest/reference/locator-strategies/) describe the supported ways to locate native controls. A stable identifier is preferred when available; another unambiguous supported locator may work. An invented identifier is never evidence that a target exists.

Later execution records will associate the chosen locator with its source/current observation. No semantics ID is required in the human's English request, and this exercise has not added a final locator schema.

Expected title/year describe what we want to verify. Observed title/year must come from the running app; no observations or verdict have been collected here.

**Stop after this exercise.** Distinguish the requested task, the business entity, the technical run profile, and the later UI locator. Our next small lesson remains the application/run profile.

## 2. Decide where the information comes from

| Source | Supplies | Example for this scenario |
| --- | --- | --- |
| Human scenario | Intended behavior and expected outcome | Search Movies for Interstellar (2014), open its details, and verify the title and year. |
| Application/run profile | Application identity, platform/provider, real backend environment, credential references, and permissions | Movies is named in the request; its technical run profile has not been configured in this notebook. |
| Approved app knowledge | Business rules and terminology | How the app identifies movies and presents title and release information. Small context can be supplied directly; larger documentation may later use RAG. |
| Executor + current observations | Supported navigation choices and actual interaction targets | Inspect the current UI hierarchy/screenshots, identify controls using supported identifiers/roles/labels, and validate any known locator hints. |
| Current-run evidence | What actually happened | UI observations, screenshots, and tool results captured during execution. None have been collected yet. |

The human should not need to provide Appium commands or selectors.

## 3. Proposed minimum input contract

An **input contract** describes the information the workflow needs. We are defining its meaning first; validated Python models come in a later lesson.

| Information | Why it matters | Current value/source |
| --- | --- | --- |
| Application/run profile reference | Selects the installed app, platform, provider, real backend, and permitted session | Not configured; Movies is the requested application name. |
| Movie identity and search query | Tells the executor what to find and how to distinguish matching titles | Query: Interstellar. Target: title Interstellar, release year 2014. |
| Expected title/year and matching rule | Makes the assertions checkable | Title Interstellar; year 2014. Exact title comparison is an explicit proposed exercise choice. |
| Starting conditions | Establishes whether login or another prerequisite is needed | Obtain from the selected profile and fresh app observations. |
| Permitted actions and limits | Bounds navigation, retries, time, and any state changes | Obtain from the run policy; this scenario requests search and viewing only. |

A profile supplies configuration so the human does not repeat it in each scenario. Missing material information triggers clarification rather than a guessed requirement. Credentials remain in their configured secret source, not in notebook cells.

## 4. Separate the proposed plan from proof

| Proposed step | Proposed check | Evidence needed |
| --- | --- | --- |
| Inspect Movies and navigate to search | The intended application and a usable search control are present | Fresh UI representation and execution context. |
| Enter Interstellar and inspect results | The target movie can be identified by title and release year 2014 | Observed result content and identifiers where available. |
| Open the matching movie | The app navigates to the intended movie's details | Navigation trace and details-screen observations. |
| Verify title and release year | The details title is Interstellar and its displayed release information establishes year 2014 | Actual UI values, screenshot/context references, and the applied comparison rules. |

This is a proposed business-level plan. The executor must ground the actual actions in the current app and its supported tools.

**Current observations: not collected. Test outcome: not evaluated.** Expected inputs and a generated plan do not establish a pass. A direct backend lookup cannot substitute for observing the requested UI behavior. Later execution will use Movies connected to its real TMDB API; no mocked catalog content will silently replace live data.

## 5. Where the human participates

If more than one observed result still matches the supplied title and year and approved app context cannot distinguish them, the executor should ask which movie was intended before continuing.

The workflow preserves the pending question and run state. After the human responds, it validates the answer and refreshes app observations before resuming. An unanswered question is not permission to guess.

For a failure work item, any required publication approval happens later, after the documenter has prepared the actual draft and evidence. Already authorized ordinary actions do not need repeated approval.

## 6. Follow the same scenario through the four agents

| Agent | Contribution |
| --- | --- |
| Executor | Interpret the request, resolve prerequisites, run permitted UI actions, and capture evidence. |
| Analyzer | Compare the agreed expected title and release year with actual UI evidence and determine the supported outcome. |
| Documenter | Assemble the result, explanation, and evidence references; prepare a failure draft when applicable. |
| Reporter | Deliver the result and publish a work item only when configured and authorized. |

The StateGraph fixes this order. Each agent returns validated information for the next stage; the classifier does not choose the next specialist after every stage. A failure still proceeds through analysis, documentation, and reporting.

## 7. Stop here and review

Before advancing, work through these questions together:

- Can we distinguish the human's requested actions from the expected outcome?
- Which details belong in the human request, and which can come from a saved profile?
- What unresolved detail would require a human question?
- What current-run evidence would establish a pass or a failure?

The next review is to confirm or revise this scenario input contract in Section 6 of the requirements. Later lessons will add small executable cells as their milestones are agreed. No future agent, server endpoint, or automation connection has been implemented in this notebook.

[Index](README.md) · [Next lesson](02_appium_execution_overview.ipynb)